# Multilayer Magnetic Sample with Variable Cobalt Thickness: Focal Plane Analysis

This notebook simulates coherent X-ray scattering (Fourier transform holography, FTH) from a complex magnetic multilayer sample featuring:
- **Varying cobalt thickness** across layers: Au(1000)/[Pt(1)Co(d)Al(1)]×Ntot
- **Three distinct magnetic regions**: stripe states (bottom), saturated state (middle), skyrmion lattice (top)
- **Multislice exit wave simulation** capturing propagation effects through the sample
- **Focal plane analysis** showing how the exit wave reconstructs at different depths
- **Interactive focus adjustment** for real-time reconstruction optimization

The workflow demonstrates how magnetic structures at different depths contribute to the holographic signal and how to selectively focus on specific regions of interest.

## 1. Import Required Libraries

In [470]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

# Add source directory to path
repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

# Imports from scattering_calculator
from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim
from scattering_calculator.interactive.interactive_widgets import cimshow
from fomocid import DATA_ROOT

## 2. Define Magnetic Material Structure with Variable Cobalt Thickness

The sample architecture is Au(1000)/[Pt(1)Co(d)Al(1)]×Ntot, where:
- The cobalt thickness d varies across layers according to a numpy array
- Ntot is the total number of magnetic repeating units
- Each unit contains Pt(1 nm), Co(variable), and Al(1 nm) layers

In [ ]:
# Define the multilayer structure with variable cobalt thickness
Ntot = 7+15+7  # Total number of magnetic repeating units [Pt(1)Co(d)Al(1)]
# Create 2D magnetization pattern for in-plane x-y plane
ny, nx = 1024,1024


# Define varying cobalt thickness (in nm) - can be constant, linear, or arbitrary variation
co_thickness_nm = np.linspace(2.0, 2.1, Ntot)  # Vary cobalt thickness from 6 to 7.0 nm

# Construct the recipe string dynamically
pt_thickness = 3  # nm
al_thickness = 1.4  # nm
au_thickness = 1000  # nm (substrate)

# Build the multilayer stack as a string
recipe_layers = [f"Au({au_thickness})/SiN(100)"]  # Au substrate
for i, co_thick in enumerate(co_thickness_nm):
    recipe_layers.append(f"Pt({pt_thickness})Co({co_thick:.2f})Al({al_thickness})")

recipe = "/".join(recipe_layers)

print(f"Sample recipe: {recipe[:100]}...")
print(f"Total cobalt thickness variations: {len(co_thickness_nm)}")
print(f"Cobalt thickness range: {co_thickness_nm.min():.2f} - {co_thickness_nm.max():.2f} nm")

# Detector and beam parameters
detector_shape = (nx,ny)
detector_pixel_size = 40e-6  # in m
detector_distance = 40e-2 # in m
real_space_pixel_size = 7.8e-9  # in m

# Sample dimensions
sample_shape = np.array([Ntot + 1, nx,ny])  # z, y, x layers for multislice

# X-ray parameters
energy_eV = 778.0  # Co L3 edge
photon_flux = 1e10

## 3. Define Magnetic States

Create a complex magnetic state with three regions:
- **Bottom N1 layers**: Vertical stripe states with period Λ
- **Middle N2 layers**: Saturated state (uniform magnetization)
- **Top N3 layers**: Skyrmion lattice aligned to stripe periodicity

In [ ]:
# Divide the sample into three magnetic regions
N1 = 7   # Number of bottom layers with stripes
N2 = 15   # Number of middle layers with saturated state
N3 = 7   # Number of top layers with skyrmions

# Stripe parameters
stripe_period = 400e-9  # Stripe period in meters
stripe_width = stripe_period / 2



# Coordinate grids
y_coords = np.arange(ny) * real_space_pixel_size
x_coords = np.arange(nx) * real_space_pixel_size

yy, xx = np.meshgrid(y_coords, x_coords, indexing='ij')

# 1. BOTTOM REGION: Vertical stripe states (mz component)
# Create vertical stripes along x-direction
stripe_pattern = np.sin(2 * np.pi * xx / stripe_period)
mz_stripes = np.tanh(stripe_pattern * 200)  # Smooth saturation

# 2. TOP REGION: Skyrmion lattice aligned to stripe periodicity
# Create hexagonal skyrmion lattice with same periodicity as stripes
skyrm_lattice_period = stripe_period *2
y_offset = (skyrm_lattice_period * np.sqrt(3) / 2)

# Generate skyrmion pattern on hexagonal lattice
skyrm_x = (xx % skyrm_lattice_period) / skyrm_lattice_period - 0.5
skyrm_y = ((yy % y_offset) / y_offset - 0.5) * (np.sqrt(3) / 2)

# Skyrmion profile: Gaussian-like magnetic textures
skyrm_radius = stripe_period / 8
skyrm_pattern = np.exp(-((skyrm_x)**2 + (skyrm_y)**2) / (2 * (skyrm_radius/stripe_period)**2))
mz_skyrmions = np.tanh((skyrm_pattern - 0.5) * 200)  # Binary skyrmion pattern

# Align skyrmions to stripes (position them on top of stripe boundaries)
mz_skyrmions = np.roll(mz_skyrmions, int((stripe_period-stripe_period/2) / (2 * real_space_pixel_size)), axis=1)

# 3. Initialize full magnetization array (layers × y × x)
magnetization_3d = np.ones((Ntot , ny, nx))

# Bottom layers: Stripes
magnetization_3d[:N1, :, :] = mz_stripes[np.newaxis, :, :]

# Middle layers: Saturated state (uniform +z magnetization)
magnetization_3d[N1:N1+N2, :, :] = -1.0

# Top layers: Skyrmions
magnetization_3d[N1+N2:N1+N2+N3, :, :] = mz_skyrmions[np.newaxis, :, :]

print(f"Magnetization array shape: {magnetization_3d.shape}")
print(f"Bottom region (stripes): layers 0-{N1-1}")
print(f"Middle region (saturated): layers {N1}-{N1+N2-1}")
print(f"Top region (skyrmions): layers {N1+N2}-{N1+N2+N3-1}")
print(f"Stripe period: {stripe_period*1e9:.1f} nm")
print(f"Skyrmion lattice period: {skyrm_lattice_period*1e9:.1f} nm")

In [ ]:
%matplotlib inline

# Visualize the three magnetic regions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Bottom: Stripes
im0 = axes[0].imshow(magnetization_3d[0, ::4, ::4], cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title(f'Bottom Region (layer 0): Vertical Stripes')
axes[0].set_ylabel('y (downsampled)')
axes[0].set_xlabel('x (downsampled)')
plt.colorbar(im0, ax=axes[0], label='mz')

# Middle: Saturated
im1 = axes[1].imshow(magnetization_3d[N1, ::4, ::4], cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title(f'Middle Region (layer {N1}): Saturated State')
axes[1].set_ylabel('y (downsampled)')
axes[1].set_xlabel('x (downsampled)')
plt.colorbar(im1, ax=axes[1], label='mz')

# Top: Skyrmions
im2 = axes[2].imshow(magnetization_3d[N1+N2, ::4, ::4], cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_title(f'Top Region (layer {N1+N2}): Skyrmion Lattice')
axes[2].set_ylabel('y (downsampled)')
axes[2].set_xlabel('x (downsampled)')
plt.colorbar(im2, ax=axes[2], label='mz')

plt.show()

## 4. Compute Exit Waves with Multislice Simulation

This section performs the coherent X-ray scattering simulation using multislice propagation, which accounts for diffraction effects as the beam propagates through each material layer.

In [ ]:
# Configure X-ray parameters
xray_config = sim.XRayConfig(
    energy=energy_eV,
    pol="CR",
    photon_flux=photon_flux,
    coherence_length=(150e-6, 150e-6),
)
xray_config.setup()

# Configure detector
detector_params = {
    "readout_noise_average": 50,
    "readout_noise_sigma": 3,
    "detector_threshold": 64e3,
    "counts_per_photon": 180,
    "quantum_efficiency": 0.85,
}

measurement_config = {
    "number_frames": 1,
    "max_counts_per_image": None,
    "exposure_time": 1.0,
}

artifacts_config = {
    "sigma_photon": 0.75,
    "photon_n_classes": 1,
    "photon_n_variants": 10,
    "photon_kernel_size": 9,
    "photon_irregularity": 2.0,
    "regenerate_photon_kernels": False,
}

beamstop_config = sim.BeamstopConfig(
    bs_method="circular",
    bs_detector_distance=0.001,
    bs_center=tuple(np.array(detector_shape) // 2),
    bs_config={
        "radius": 0.1e-3,  # Small radius (not blocking much)
        "angle": 0.0,
        "sigma": 0.01e-3,
        "ellipticity": (1.0, 1.0),
        "roughness": 0.0,
        "roughness_modes": (3, 9),
        "wire_width": 0.0,
        "wire_bend": 0.0,
        "antialias": 1,
        "seed": None,
    },
)

detector_config = sim.DetectorConfig(
    pixel_size=detector_pixel_size,
    shape=detector_shape,
    sample_to_detector_distance=detector_distance,
    detector_center=tuple(np.array(detector_shape) // 2),
    detector_params=detector_params,
    measurement_config=measurement_config,
    artifacts_config=artifacts_config,
    beamstop_config=beamstop_config,
)
detector_config.setup()

# Visualize beamstop to ensure detector is fully initialized
print("Initializing detector geometry...")
# This step ensures all detector layout attributes (including q-space coordinates) are computed
try:
    detector_config.visualize_beamstop()
except Exception as e:
    print(f"Beamstop visualization skipped (this is OK): {type(e).__name__}")

print("Detector configuration completed.")

# Explicitly initialize detector coordinate systems for hologram detection
# Access detector layout properties to trigger lazy initialization of q-space coordinates
_ = detector_config.detector_layout.detx
_ = detector_config.detector_layout.dety
print(f"Detector initialized: shape={detector_config.detector_layout.detector_shape}, center={detector_config.detector_layout.detector_center}")

# Configure sample
sample_config = sim.SampleConfig(
    recipe=recipe,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    xray_config=xray_config,
    sample_name="multilayer_focus_tutorial",
)
sample_config.setup()

# Assign magnetization to sample
# Create full 3D magnetization array: (Nz, Ny, Nx, 3) for (mx, my, mz)
magnetization = np.zeros((*magnetization_3d.shape, 3))
magnetization[..., 0] = 0  # mx = 0 (no x-component)
magnetization[..., 1] = np.sqrt(np.clip(1 - np.abs(magnetization_3d)**2, 0, 1))  # my (from perpendicular constraint)
magnetization[..., 2] = magnetization_3d  # mz (z-component from patterns)

# Replicate magnetization pattern to match total number of sample layers
# The first magnetization_3d.shape[0] layers get the patterns; remaining Au/interface layers are non-magnetic
if sample_shape[0] > magnetization_3d.shape[0]:
    # Pad with zeros for non-magnetic layers (Au substrate, etc.)
    padding = ((0, sample_shape[0] - magnetization_3d.shape[0]), (0, 0), (0, 0), (0, 0))
    magnetization = np.pad(magnetization, padding, mode='constant', constant_values=0)
print(sample_shape, magnetization.shape)

sample_config.assign_magnetic_pattern(magnetization)

# Configure apertures (object hole + reference holes for FTH)
aperture_config = sim.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=sample_config.sample_structure.sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    aperture_thicknesses=sample_config.sample_structure.layer_thicknesses,
    use_roi=True,
    aperture_config={
        "apertures_type": ["OH"],
        "apertures_radius": [1055e-9],#, 3e-9, 2e-9],
        "apertures_center": [(0.0, 0.0)],#, (-2040e-9, -2025e-9), (2025e-9, -2000e-9)],
        "apertures_sigma": [4e-9],# 2e-9, 2e-9],
        "apertures_angle": [0.0],#, 0.0, 0.0],
        "apertures_ellipticity": [1.0],#, 1.0, 1.0],
        "apertures_roughness": [0.0],#, 0.02, 0.02],
        "apertures_roughness_modes": [(0, 0)],#, (3, 10), (3, 10)],
        "apertures_seed": [1],#, 2, 3],
        "apertures_top_radius_factor": [1.0],#, 1.9, 1.75],
        "aperture_taper_depth": 1400e-9,
        "thickness_OH": 1000e-9,
    },
)
aperture_config.setup()
aperture_mask = aperture_config.return_aperture()
sample_config.assign_aperture_mask(aperture_mask)

# Prepare scalar refractive index for faster computation
sample_config.sample_structure.calculate_final_scalar_refractive_index(
    pol="CR",
    use_aperture_roi=True,
    compact=True,
    lazy=True,
)

print("X-ray, detector, sample, and aperture configurations completed.")

In [ ]:
aperture_config.visualize_aperture()

In [ ]:
# Configure illumination
illumination_config = sim.IlluminationConfig(
    XRayConfig=xray_config,
    shape=tuple(sample_config.sample_structure.sample_shape[1:]),
    real_space_pixel_size=real_space_pixel_size,
    illumination_function="gaussian",
    illumination_config={
        "center": np.array([0.0, 0.0]),
        "distance": 1e-3,
        "fwhm": 50e-6,
        "alpha_beam": (0.0, 0.0),
    },
)
illumination_config.setup()

# Create hologram config for storing results
hologram_config = sim.HologramConfig(
    sample_x=sample_config.sample_structure.x,
    sample_y=sample_config.sample_structure.y,
    detector_layout=detector_config.detector_layout,
)

# Perform multislice simulation for both CR and CL polarizations
print("Starting multislice simulation...")

for pol_idx, pol in enumerate(["CR", "CL"]):
    print(f"  Computing {pol} polarization...")
    
    detector_config.detector_params["noise_seed"] = 1000 + pol_idx
    illumination_config.update_polarization(pol)
    
    # Configure propagator with multislice enabled
    propagator_config = sim.SamplePropagatorConfig(
        SampleConfig=sample_config,
        IlluminationConfig=illumination_config,
        propagator_method="Scalar",
        propagator_config={
            "propagator_method": "Scalar",
            "propagate": True,  # Enable multislice propagation
            "jones_apply_zero_order_phase": True,
            "scalar_apply_zero_order_phase": True,
            "scalar_refractive_index_lazy": True,
            "propagation_padding_px": 20,
            "propagation_padding_mode": "edge",
            "propagation_absorber_width_px": 10,
            "propagation_absorber_strength": 6.0,
            "propagation_absorber_profile": "cosine",
            "multislice_propagation_roi": True,
            "multislice_propagation_roi_padding_px": 20,
            "multislice_propagation_roi_merge_overlaps": True,
        },
    )
    propagator_config.setup()
    detector_config.setup()
  
    ## This is the resolution we will have thanks to the detector
    real_space_pixel_size_fake = (
        detector_config.calc_realspace_resolution(illumination_config.beam_params) / 2  # /2 for 2× oversampling
    )
    print(f"Estimated real-space resolution from detector: {real_space_pixel_size_fake*1e9:.2f} nm")


    # Detect hologram
    detector_config.assign_propagated_wavefront(propagator_config)
    detector_config.detect_hologram()  # add Poisson shot noise for the "detected" version
    detector_config.hologram_exp.gnomonic_projection()  # correct for curved Ewald sphere geometry


    detector_config.detect_hologram()
    
    # Store results
    exit_wave = propagator_config.return_scalar_wavefield()
    hologram_config.add_exit_waves({pol: exit_wave})
    hologram_config.add_holograms({pol: detector_config.return_ideal_hologram()}, source="ideal")
    hologram_config.add_holograms({pol: detector_config.return_detected_hologram()}, source="detected")

# Compute differences and reconstructions
hologram_config.compute_differences()
hologram_config.compute_sums()
hologram_config.compute_reconstructions()

print("Multislice simulation completed.")

## 5. Visualize Exit Waves and Holograms

In [ ]:
%matplotlib inline

# Extract and visualize exit waves
cr_exit = hologram_config.exit_waves["CR"][:,400:-400,400:-400]
cl_exit = hologram_config.exit_waves["CL"][:,400:-400,400:-400]

# Visualize exit waves
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# CR exit wave
im00 = axes[0, 0].imshow(np.abs(cr_exit[0,::4, ::4]), cmap='viridis')
axes[0, 0].set_title('CR Exit Wave Amplitude')
plt.colorbar(im00, ax=axes[0, 0])

im01 = axes[0, 1].imshow(np.angle(cr_exit[0,::4, ::4]), cmap='hsv')
axes[0, 1].set_title('CR Exit Wave Phase')
plt.colorbar(im01, ax=axes[0, 1])

im02 = axes[0, 2].imshow(np.abs(cr_exit - cl_exit)[0,::4, ::4], cmap='viridis')
axes[0, 2].set_title('(CR - CL) Amplitude')
plt.colorbar(im02, ax=axes[0, 2])

# CL exit wave
im10 = axes[1, 0].imshow(np.abs(cl_exit[0,::4, ::4]), cmap='viridis')
axes[1, 0].set_title('CL Exit Wave Amplitude')
plt.colorbar(im10, ax=axes[1, 0])

im11 = axes[1, 1].imshow(np.angle(cl_exit[0,::4, ::4]), cmap='hsv')
axes[1, 1].set_title('CL Exit Wave Phase')
plt.colorbar(im11, ax=axes[1, 1])

# XMCD signal
xmcd_ratio = (cr_exit + 1e-30) / (cl_exit + 1e-30)
im12 = axes[1, 2].imshow(np.real(np.log(xmcd_ratio))[0,::4, ::4], cmap='RdBu_r')
axes[1, 2].set_title('XMCD Signal (log amplitude)')
plt.colorbar(im12, ax=axes[1, 2])

plt.show()

# Visualize ideal holograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ideal_cr = hologram_config.ideal_holograms["CR"]
ideal_cl = hologram_config.ideal_holograms["CL"]
ideal_diff = hologram_config.ideal_holograms["diff"]

im0 = axes[0].imshow(ideal_cr[0,::4, ::4], cmap='viridis')
axes[0].set_title('Ideal CR Hologram')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(ideal_cl[0,::4, ::4], cmap='viridis')
axes[1].set_title('Ideal CL Hologram')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(ideal_diff[0,::4, ::4], cmap='RdBu_r', vmin=-np.percentile(np.abs(ideal_diff), 99), 
                      vmax=np.percentile(np.abs(ideal_diff), 99))
axes[2].set_title('Ideal Difference (CR - CL)')
plt.colorbar(im2, ax=axes[2])

plt.show()

print(f"Exit wave shape: {cr_exit.shape}")
print(f"Hologram shape: {ideal_cr.shape}")

## 6. Propagate Exit Wave to Different Focal Planes

Free-space propagation of the exit wave to different depths (bottom, middle, top layers) to show how focus location affects the reconstruction quality.

In [ ]:
def angular_spectrum_propagate(field, wavelength, pixel_size, propagation_distance, refractive_index=1.0):
    """Propagate field using angular spectrum method.
    
    Parameters:
    -----------
    field : ndarray (2D complex)
        Input field
    wavelength : float
        X-ray wavelength in vacuum (in meters)
    pixel_size : float
        Pixel size in meters
    propagation_distance : float
        Distance to propagate (positive = away from sample)
    refractive_index : float, optional
        Complex refractive index of the medium for propagation.
        Default is 1.0 (vacuum). For propagation through magnetic material,
        use the refractive index of that material (typically 1.0 - δ - iβ for X-rays).
        
    Returns:
    --------
    propagated_field : ndarray (2D complex)
        Propagated field
        
    Notes:
    ------
    When propagating through a medium with refractive index n ≠ 1, the wavevector
    is modified: k_medium = k_vacuum * n. This gives physically accurate propagation
    distances accounting for the refractive properties of the material.
    """
    if propagation_distance == 0:
        return field.copy()
    
    ny, nx = field.shape
    
    # Frequency grids
    fy = np.fft.fftfreq(ny, pixel_size)
    fx = np.fft.fftfreq(nx, pixel_size)
    Fy, Fx = np.meshgrid(fy, fx, indexing='ij')
    
    # Angular spectrum operator with refractive index correction
    # In a medium: k_medium = k_vacuum * refractive_index
    k = 2 * np.pi / wavelength * refractive_index
    kz = np.sqrt(np.clip(k**2 - (2*np.pi*Fy)**2 - (2*np.pi*Fx)**2, 0, None))
    
    # Propagation phase
    phase = np.exp(1j * kz * propagation_distance)
    
    # Apply propagation
    field_freq = np.fft.fft2(field)
    propagated_freq = field_freq * phase
    propagated_field = np.fft.ifft2(propagated_freq)
    
    return propagated_field

# Define focal planes
wavelength = xray_config.beam_params.wavelength

# Focal planes at different layer regions
# Calculate approximate z-distances to each region
z_bottom = np.sum(sample_config.sample_structure.layer_thicknesses[-(N3 // 2):])  # Middle of bottom region
z_middle = np.sum(sample_config.sample_structure.layer_thicknesses[-(N3 + N2 // 2):])  # Middle of middle region
z_top = np.sum(sample_config.sample_structure.layer_thicknesses[-(N3 + N2 + N1 // 2):])  # Middle of top region

print(z_bottom, z_middle, z_top)

focal_planes = {
    'zero': 0.0,
    'bottom': z_bottom,
    'middle': z_middle,
    'top': z_top,
}

print(f"Focal plane depths:")
for name, z in focal_planes.items():
    print(f"  {name}: {z*1e9:.1f} nm from sample entrance")

# Propagate exit waves to focal planes
propagated_waves = {}
for pol in ["CR", "CL"]:
    propagated_waves[pol] = {}
    exit_wave = hologram_config.exit_waves[pol]
    
    for plane_name, plane_z in focal_planes.items():
        # Distance to propagate: from sample exit to focal plane
        # Negative because we're propagating backwards from exit wave
        prop_distance = plane_z - 0*sample_config.sample_structure.layer_thicknesses[-1]
        exit_wave=np.squeeze(exit_wave)
        propagated = angular_spectrum_propagate(
            exit_wave, wavelength, real_space_pixel_size, prop_distance,(sample_config.sample_structure.return_layer_refractive_indices()[-1,0])
        )
        propagated_waves[pol][plane_name] = propagated

print("Exit wave propagation to focal planes completed.")

In [ ]:
wavelength,prop_distance,real_space_pixel_size

## 7. Visualize Reconstructions at Multiple Focal Planes

Display reconstructions at each focal plane showing difference and sum contrasts. This demonstrates how the magnetic contrast from different layers (stripes, saturated, skyrmions) becomes visible when focusing at their respective depths.

In [ ]:
roicrop=np.s_[420:-370, 550:-250]  # Crop edges for better visualization
#roicrop=np.s_[200:-200, 200:-200]
roicrop=np.s_[400:-500, 470:-400]

# Compute difference and sum propagated waves at each focal plane
fig, axes = plt.subplots(4, 3, figsize=(16, 16))

for row, plane_name in enumerate(['zero','bottom', 'middle', 'top']):
    cr_prop = propagated_waves["CR"][plane_name][roicrop]
    cl_prop = propagated_waves["CL"][plane_name][roicrop]
    
    # Difference (CR - CL)
    diff_prop = cr_prop - cl_prop
    
    # Sum (CR + CL)
    sum_prop = cr_prop + cl_prop
    
    # XMCD ratio
    ratio_prop = (cr_prop + 1e-30) / (cl_prop + 1e-30)
    
    # Plot amplitude of difference
    vmax_diff = np.percentile(np.abs(diff_prop), 99)
    im0 = axes[row, 0].imshow(np.abs(diff_prop[::4, ::4]), cmap='viridis', vmax=vmax_diff)
    axes[row, 0].set_title(f'{plane_name.capitalize()} Plane: |CR - CL| Amplitude')
    plt.colorbar(im0, ax=axes[row, 0])
    
    # Plot sum intensity
    vmax_sum = np.percentile(np.abs(sum_prop), 99)
    im1 = axes[row, 1].imshow(np.abs(sum_prop[::4, ::4]), cmap='viridis', vmax=vmax_sum)
    axes[row, 1].set_title(f'{plane_name.capitalize()} Plane: |CR + CL| Intensity')
    plt.colorbar(im1, ax=axes[row, 1])
    
    # Plot XMCD signal
    im2 = axes[row, 2].imshow(np.real(np.log(ratio_prop))[::4, ::4], cmap='RdBu_r')
    axes[row, 2].set_title(f'{plane_name.capitalize()} Plane: XMCD (log amplitude)')
    plt.colorbar(im2, ax=axes[row, 2])

plt.show()

print("Focal plane analysis completed.")

In [ ]:
# Compute FTH reconstructions from propagated hologram waves
# Create holograms from propagated waves and reconstruct

def fth_reconstruct(hologram):
    """Perform 2D FFT-based FTH reconstruction."""
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(hologram)))

# Visualize FTH reconstructions at different focal planes
fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for row, plane_name in enumerate(['bottom', 'middle', 'top']):
    cr_prop = propagated_waves["CR"][plane_name]
    cl_prop = propagated_waves["CL"][plane_name]
    
    # Create difference and sum holograms
    diff_holo = cr_prop - cl_prop
    sum_holo = cr_prop + cl_prop
    
    # Reconstruct
    diff_recon = fth_reconstruct(diff_holo)
    sum_recon = fth_reconstruct(sum_holo)
    
    # Crop to object hole region for display
    crop_size = 256
    cy, cx = diff_recon.shape[0] // 2, diff_recon.shape[1] // 2
    diff_crop = diff_recon[cy-crop_size:cy+crop_size, cx-crop_size:cx+crop_size]
    sum_crop = sum_recon[cy-crop_size:cy+crop_size, cx-crop_size:cx+crop_size]
    
    # Plot difference reconstruction
    vmax_diff = np.percentile(np.abs(diff_crop), 99)
    im0 = axes[row, 0].imshow(np.abs(diff_crop), cmap='viridis', vmax=vmax_diff)
    axes[row, 0].set_title(f'{plane_name.capitalize()}: |FTH(CR - CL)|')
    plt.colorbar(im0, ax=axes[row, 0])
    
    # Plot phase of difference
    im1 = axes[row, 1].imshow(np.angle(diff_crop), cmap='hsv')
    axes[row, 1].set_title(f'{plane_name.capitalize()}: ∠FTH(CR - CL)')
    plt.colorbar(im1, ax=axes[row, 1])
    
    # Plot sum reconstruction (intensity)
    im2 = axes[row, 2].imshow(np.abs(sum_crop), cmap='viridis')
    axes[row, 2].set_title(f'{plane_name.capitalize()}: |FTH(CR + CL)|')
    plt.colorbar(im2, ax=axes[row, 2])

plt.show()

print("FTH reconstruction at focal planes completed.")

## 8. Interactive Focal Plane Adjustment

Use interactive widgets to adjust the focal plane in real-time and observe how the reconstruction changes. This allows exploration of the depth-dependent contrast in the sample.

In [ ]:
from ipywidgets import interactive, FloatSlider, fixed
import ipywidgets as widgets

# Store the ideal difference and sum holograms for interactive adjustment
ideal_diff_holo = hologram_config.ideal_holograms["diff"]
ideal_sum_holo = hologram_config.ideal_holograms["sum"]

def interactive_focus_reconstruction(focal_distance_nm):
    """Compute and display FTH reconstruction at specified focal distance.
    
    Parameters:
    -----------
    focal_distance_nm : float
        Distance from sample exit to focal plane (in nm)
    """
    
    # Convert to meters
    focal_distance = focal_distance_nm * 1e-9
    
    # Propagate holograms to focal plane
    diff_prop = angular_spectrum_propagate(
        ideal_diff_holo, wavelength, real_space_pixel_size, focal_distance
    )
    sum_prop = angular_spectrum_propagate(
        ideal_sum_holo, wavelength, real_space_pixel_size, focal_distance
    )
    
    # Reconstruct via FFT
    diff_recon = fth_reconstruct(diff_prop)
    sum_recon = fth_reconstruct(sum_prop)
    
    # Crop to object hole for display
    crop_size = 256
    cy, cx = diff_recon.shape[0] // 2, diff_recon.shape[1] // 2
    diff_crop = diff_recon[cy-crop_size:cy+crop_size, cx-crop_size:cx+crop_size]
    sum_crop = sum_recon[cy-crop_size:cy+crop_size, cx-crop_size:cx+crop_size]
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Plot difference amplitude
    vmax_diff = np.percentile(np.abs(diff_crop), 99)
    im0 = axes[0].imshow(np.abs(diff_crop), cmap='viridis', vmax=vmax_diff)
    axes[0].set_title(f'FTH Difference |CR - CL| at z = {focal_distance_nm:.1f} nm')
    plt.colorbar(im0, ax=axes[0], label='Amplitude')
    
    # Plot difference phase
    im1 = axes[1].imshow(np.angle(diff_crop), cmap='hsv')
    axes[1].set_title(f'Phase at z = {focal_distance_nm:.1f} nm')
    plt.colorbar(im1, ax=axes[1], label='Phase (rad)')
    
    # Plot sum intensity
    vmax_sum = np.percentile(np.abs(sum_crop), 99)
    im2 = axes[2].imshow(np.abs(sum_crop), cmap='viridis', vmax=vmax_sum)
    axes[2].set_title(f'FTH Sum |CR + CL| at z = {focal_distance_nm:.1f} nm')
    plt.colorbar(im2, ax=axes[2], label='Intensity')
    
    # Add reference lines for magnetic regions
    for ax in axes:
        ax.axhline(crop_size, color='red', linestyle='--', alpha=0.3, linewidth=1)
        ax.axvline(crop_size, color='red', linestyle='--', alpha=0.3, linewidth=1)
    
    plt.tight_layout()
    plt.show()
    
    # Compute and display contrast metrics
    contrast_amp = np.std(np.abs(diff_crop))
    contrast_phase = np.std(np.angle(diff_crop))
    
    print(f"Focal plane at z = {focal_distance_nm:.1f} nm:")
    print(f"  Amplitude contrast (std): {contrast_amp:.4f}")
    print(f"  Phase contrast (std): {contrast_phase:.4f}")

# Define focal distance slider
# Range: from bottom region to top region
z_range_nm = (focal_planes['top'] - focal_planes['bottom']) * 1e9
z_min = -z_range_nm * 0.5  # Start before bottom
z_max = z_range_nm * 1.5   # End after top

focal_slider = FloatSlider(
    value=0,
    min=z_min,
    max=z_max,
    step=10,
    description='Focus position (nm):',
    orientation='horizontal',
    layout=widgets.Layout(width='400px')
)

# Create interactive widget
print("Interactive focal plane adjustment:")
print(f"Adjust the slider to focus at different depths in the sample.")
print(f"Range: {z_min:.0f} to {z_max:.0f} nm from sample exit")

interactive_focus = interactive(interactive_focus_reconstruction, focal_distance_nm=focal_slider)
display(interactive_focus)

## Summary

This notebook demonstrates:

1. **Complex multilayer structure**: Au(1000)/[Pt(1)Co(d)Al(1)]×Ntot with variable cobalt thickness
2. **Multi-region magnetic states**: 
   - Vertical stripes in bottom N1 layers
   - Saturated state in middle N2 layers  
   - Skyrmion lattice in top N3 layers, aligned to stripe periodicity
3. **Multislice exit wave simulation**: Full propagation calculation accounting for diffraction at each layer
4. **Focal plane analysis**: Demonstrates how different layers contribute to the holographic signal at different depths
5. **Interactive focusing**: Real-time adjustment of focal plane to optimize reconstruction of specific magnetic features

**Key insights**:
- Different magnetic structures at different depths produce different contrast at their respective focal planes
- The interactive slider allows exploration of depth-dependent sensitivity in FTH
- Multislice propagation captures the complexity of wave propagation through the magnetic multilayer structure

## Tips for Customization

**Adjust magnetic structure**:
- Change `Ntot` to increase/decrease total number of layers
- Modify `co_thickness_nm = np.linspace(...)` to create different thickness profiles (constant, quadratic, random, etc.)
- Adjust `N1`, `N2`, `N3` to change the relative sizes of the three magnetic regions

**Modify magnetic patterns**:
- Change `stripe_period` to vary the stripe wavelength
- Adjust the skyrmion pattern by modifying `skyrm_radius` and the lattice periodicity
- Combine stripe and skyrmion patterns or create new custom patterns

**Simulation parameters**:
- Increase `detector_shape` for higher resolution (more computationally expensive)
- Change `propagate=True/False` to enable/disable multislice propagation
- Adjust `propagation_padding_px` to control the balance between accuracy and computation time
- Modify `energy_eV` to simulate different X-ray energies (e.g., resonant vs. non-resonant)

**Focal plane analysis**:
- Use the interactive slider to identify the optimal focal distance for each magnetic region
- Compare amplitude and phase contrast across different focal planes
- Study how the skyrmion/stripe resolution changes with focal position